## Challenge Statement

A chemical species is suspended in a fluid. As time passes, the species diffuses through the fluid while undergoing a reaction that causes it to decay. The concentration $\phi(x,t)$ is governed by the one-dimensional diffusion-reaction equation

$$\frac{\partial \phi}{\partial t} = \nu \frac{\partial^2 \phi}{\partial x^2} - k\phi + s(x,t),$$

where $\nu$ is the diffusion coefficient, $k$ is the reaction rate, and $s(x,t)$ describes a source of the chemical species (reality is three-dimensional though, take this as a simplification for learning purposes!)

In this challenge, the diffusion coefficient $\nu$ and reaction rate $k$ will be given as inputs. Your goal is to use QSVT to determine the concentration profile after one time step.

In order to do this, we need to express the PDE in discrete terms. We discretize the fluid into four interior spatial points separated by

$$h = 0.2.$$

The concentration field at a given time can therefore be represented by the vector

$$\phi = \begin{pmatrix} \phi_1 \\ \phi_2 \\ \phi_3 \\ \phi_4 \end{pmatrix},$$

where $\phi_i$ is the concentration at the $i$-th spatial point.

Using a central finite-difference approximation, the second spatial derivative is represented by

$$D^2 = \frac{1}{h^2} \begin{pmatrix} -2 & 1 & 0 & 0 \\ 1 & -2 & 1 & 0 \\ 0 & 1 & -2 & 1 \\ 0 & 0 & 1 & -2 \end{pmatrix}.$$

We evolve the system through a time step

$$\Delta t = 0.01$$

using the backward Euler method. The diffusion–reaction equation can then be written as the linear system

$$A\phi_{new} = b,$$

where

$$A = (1 + k\Delta t)I - \nu\Delta t D^2$$

and

$$b = \phi_{old} + \Delta t \, s.$$

In this challenge, $h$, $\Delta t$, $\phi_{old}$, and $s$ are fixed and provided in the starter code. Only $\nu$ and $k$ change between experiments.

Classically, we could obtain the new concentration simply by calculating

$$\phi_{new} = A^{-1}b.$$

You, however, will use Quantum Singular Value Transformation (QSVT) to approximately implement this inverse.

## Challenge Code

Complete the function `qsvt_diffusion` that uses QSVT to approximate the solution to the next time step of the differential equation above. See figure below for reference.

You will need to find some phase angles for QSVT that approximate the polynomial $C/x$, for some real number $C$.

Please put your `C` and `qsvt_angles` in your code, as they will be used by the grader.

You have some free space for helper functions, use it wisely!

Your solution should:

1. Construct the matrix $A$ corresponding to the supplied values of $\nu$ and $k$.
2. Construct an appropriate polynomial approximation to the inverse $C/x$.
3. Determine the QSVT phase angles corresponding to this polynomial and specify the value of $C$.
4. Use QSVT to approximately apply the inverse of $A$.

The grader will use your value of $C$ to rescale your output for the QSVT circuit and compare with the true solution. The test cases should be lenient enough that reasonable QSVT phases should work, but please note that the grader is looking for fidelity > 0.99. Make sure to refine your angles properly if some test cases fail!

## Input

As input to this problem, you are given a list containing the diffusion coefficient `nu` (float) and the reaction rate `k` (float).

## Output

This code will output a `np.array(float)` with the solution to the concentration vector after one time step.

## Test cases

The following public test cases are available to you. Note that there are additional hidden test cases that we use to verify that your code is valid in full generality.

```text
test_input: [0.2, 0.5]
expected_output: [0.27959746, 0.42869397, 0.56892447, 0.64371709]

test_input: [1.0, 1.2]
expected_output: [0.28550356, 0.4595721 , 0.59350887, 0.59584262]

```

If your solution matches the correct one within the given tolerance specified in check (in this case, it's a fidelity greater than 0.99 of the approximate solution with respect to the analytical solution), the output will be "Success!" Otherwise, you will receive an "Incorrect" prompt.

Good luck!

In [ ]:
import json
import pennylane as qp
import pennylane.numpy as np

# DO NOT CHANGE THESE VARIABLES LATER IN THE CODE

phi = np.array([0.03, 0.04, 0.07, 0.09])
source = np.array([1., 2., 1., 0.5])
dt = 0.01
h = 0.2


def _fit_inverse_poly(lo, degree, margin=0.5):
    ys = np.linspace(lo, 1, 400)
    powers = list(range(1, degree + 1, 2))
    M = np.array([ys ** p for p in powers]).T
    M = M * ys[:, None]
    ones = np.ones_like(ys)
    coeffs, *_ = np.linalg.lstsq(M, ones, rcond=None)
    poly = np.zeros(degree + 1)
    for c, p in zip(coeffs, powers):
        poly[p] = c
    grid = np.linspace(-1, 1, 2000)
    top = np.max(np.abs(np.polyval(poly[::-1], grid)))
    scale = margin / top
    return poly * scale, scale


# Odd polynomial ~ C/x on [0.1, 1]
_poly, C = _fit_inverse_poly(0.1, 33)
phi_qsvt = qp.poly_to_angles(_poly, "QSVT")

b = phi + dt * source

n_pts = len(phi)

# Feel free to write any helper functions here
device = qp.device("default.qubit", wires=4)

device = qp.device("default.qubit", wires=4)


@qp.qnode(device)
def qsvt_diffusion(nu, k):

    """
    Implements QSVT to find the concentration phi after one time step

    Args:
    - nu (float): The diffusion coefficient
    - k (float): The reaction rate

    Returns:
    - (np.array): The concentation vector on the next time step.
    """


    # Write your QSVT code for matrix inversion here
    D2 = np.zeros((n_pts, n_pts))
    for i in range(n_pts):
        D2[i, i] = -2
        if i > 0:
            D2[i, i - 1] = 1
        if i < n_pts - 1:
            D2[i, i + 1] = 1
    D2 = D2 / h ** 2
    A = (1 + k * dt) * np.eye(n_pts) - nu * dt * D2
    A_norm = A / np.linalg.norm(A, 2)

    qp.StatePrep(b / np.linalg.norm(b), wires=[2, 3])

    # Hadamard test on wire 0 gives Re of QSVT
    wr = [1, 2, 3]
    be = qp.BlockEncode(A_norm, wires=wr)
    up = [qp.PCPhase(a, 4, wr) for a in phi_qsvt]
    dn = [qp.PCPhase(-a, 4, wr) for a in phi_qsvt]
    qp.Hadamard(wires=0)
    qp.ctrl(qp.QSVT, 0, control_values=0)(be, up)
    qp.ctrl(qp.QSVT, 0, control_values=1)(be, dn)
    qp.Hadamard(wires=0)

    return qp.state()
    # Return the state

# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    out_state = qsvt_diffusion(*ins)[:4]
    rescaled_out= out_state * np.linalg.norm(b) / C
    output = rescaled_out/ np.linalg.norm(rescaled_out)

    return str(np.real(output).tolist())

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    fidelity = np.dot(solution_output, expected_output)
    assert fidelity >=0.99, "Your circuit doesn't look quite right "


# These are the public test cases
test_cases = [
    ('[0.2,0.5]', '[0.27959746, 0.42869397, 0.56892447, 0.64371709]'),
    ('[1.0, 1.2]', '[0.28550356, 0.4595721 , 0.59350887, 0.59584262]')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")